In [ ]:
import requests
import pandas as pd
import json

In [ ]:
# the query protein according to UNIPROT identifier
query = "VCAM1_HUMAN"

In [ ]:
# Use the UniProt-API to write a function that gets the ProteinID or the Protein name

def convert_protein_ID_Name(uniprotID_or_name): # e.g. can be P19320 or VCAM1_HUMAN
    
    uniprot_api_url = "https://rest.uniprot.org/uniprotkb" 
    format = "json"
    uniprot_request_url = f"{uniprot_api_url}/{uniprotID_or_name}?format={format}"
    uniprot_json = requests.get(uniprot_request_url).text
    
    uniprot_dict = json.loads(uniprot_json) # convering to dictionary

    if "_" in uniprotID_or_name:
        value = uniprot_dict["primaryAccession"]
    else:
        value = uniprot_dict['uniProtkbId']
        
    return (value)

# example
id_or_name = convert_protein_ID_Name("Q13740")
print(id_or_name)

In [ ]:
# GETTING THE DATA FROM the PSICQUIC-API

psicquic_api_url = "http://www.ebi.ac.uk/Tools/webservices/psicquic/intact/webservices"

version = "current"
method = "interactor"
protein = query
format = "tab25"

psicquic_request_url = f"{psicquic_api_url}/{version}/search/{method}/{protein}?format={format}"

psicquic_data = requests.get(psicquic_request_url).text

# x = len(psicquic_data.readlines())

print(psicquic_request_url)

In [ ]:
# Getting the data into a pandas df

# Split each row into a list of columns based on PSI-MI TAB 2.5 format
psicquic_columns = ['Unique identifier for interactor A', 
                    'Unique identifier for interactor B', 
                    'Alternative identifier for interactor A', 
                    'Alternative identifier for interactor B', 
                    'Aliases for A', 
                    'Aliases for B', 
                    'Interaction detection methods', 
                    'First author', 
                    'Identifier of the publication', 
                    'NCBI Taxonomy identifier for interactor A', 
                    'NCBI Taxonomy identifier for interactor B', 
                    'Interaction types', 
                    'Source databases', 
                    'Interaction identifier(s)', 
                    'Confidence score']

psicquic_rows = [row.split('\t') for row in psicquic_data.split('\n')]

# Create a pandas DataFrame from the list of rows and columns
psicquic_df = pd.DataFrame(psicquic_rows, columns=psicquic_columns)

psicquic_df.shape


# Now use the UniProt API to get the name of the protein of the interactors Or find other method

psicquic_df_A = psicquic_df[['Aliases for A', 'Aliases for B']]
psicquic_df.head(5)

In [ ]:
# GETTING DATA FROM the STRING-API

string_api_url = "https://string-db.org//api"

output_format = "json"
method = "interaction_partners"

request_url = "/".join([string_api_url, output_format, method])

params = {
    "identifiers": query,
    "species": 9606,
    "required_score": 0.2,
    "limit": 1000000 
}

response = requests.post(request_url, data = params)

string_data = response.text

print(string_data)

In [ ]:
# Getting the STRING-API data into a pandas df

df = pd.read_json(string_data)
df.shape

print(df)

In [ ]:
# GETTING THE DATA FROM the HIPPIE-API

hippie_api_url = "http://cbdm-01.zdv.uni-mainz.de/~mschaefer/hippie/queryHIPPIE.php"

protein_to_query = "VCAM1_HUMAN"
layer = 1 #to query protein within input set (0) or against all HIPPIE proteins (1, default)
threshold = 0 #confidence threshold, default is 0
format = "browser" #this generates a tab seperated text file (other interesting input types: "mitab", "browser")

hippie_request_url = f"{hippie_api_url}?proteins={protein_to_query}&layers={layer}&conf_thres={threshold}&out_type={format}"

hippie_response = requests.get(hippie_request_url).text

print(hippie_response)

print(hippie_request_url)

# the table is written in javascript 


In [ ]:
# GETTING THE DATA FROM HIPPIE through WEBSCRAPING
import requests
import json
import re
from bs4 import BeautifulSoup

protein = 'VCAM1_HUMAN'

url = "http://cbdm-01.zdv.uni-mainz.de/~mschaefer/hippie/query.php?s="+str(protein)

payload = {}
headers = {
'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7', 
'Accept-Language': 'nl-NL,nl;q=0.9,en-US;q=0.8,en;q=0.7,fr;q=0.6' ,
'Connection': 'keep-alive', 
'Referer': 'http://cbdm-01.zdv.uni-mainz.de/~mschaefer/hippie/',
'Upgrade-Insecure-Requests': '1' ,
'User-Agent': 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Mobile Safari/537.36' 
}

response = requests.request("GET", url, headers=headers, data=payload)
soup = BeautifulSoup(response.text, "html.parser")
table = soup.find('tbody') # already skipped the columns names

rows = table.find_all('tr')

interactions = []
for idx, row in enumerate(rows):
    data = row.find_all("td")
    interaction = {
    "Interactor": data[0].text,
    "EntrezGeneID": data[1].text,
    "GeneSymbol": data[2].text,
    "Score": data[3].text
    }
    interactions.append(interaction) 
   

print(interactions)
print(len(interactions))
print(json.dumps(interactions, indent=4))

In [ ]:
# GETTING THE DATA FROM THE APID db by WEBSCRAPING

import requests
import json
import re
from bs4 import BeautifulSoup


#Function gets protein id from name value. (Example: VCAM1_HUMAN -> P19320). We will need this protein ID to generate our table.
def extract_protein_id(name):
    url = "http://cicblade.dep.usal.es:8080/APID/searchProtein.action" # Endpoint to search for protein by name

    payload = 'proteinName='+str(name)+'&taxon=0'
    headers = {
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'Accept-Language': 'en-US,en;q=0.9',
    'Cache-Control': 'max-age=0',
    'Connection': 'keep-alive',
    'Content-Type': 'application/x-www-form-urlencoded',
    'Cookie': 'JSESSIONID=086030D12C94A8248DA2B5B9A84C16FA; _ga=GA1.2.581619552.1696441740; _gid=GA1.2.818200239.1696441740; _gat=1; _ga_7JSDHY18SK=GS1.2.1696441740.1.1.1696443448.0.0.0',
    'Origin': 'http://cicblade.dep.usal.es:8080',
    'Referer': 'http://cicblade.dep.usal.es:8080/APID/init.action',
    'Upgrade-Insecure-Requests': '1',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'
    }

    response = requests.request("POST", url, headers=headers, data=payload) # Execute HTTP request, response is stored in 'response'
    soup = BeautifulSoup(response.text, "html.parser") #Parse response.text (which is HTML) with a HTML parser
    table = soup.find('table', id="proteins") #use soup.find function to find a table in the HTML with id "proteins" (You can get this value by checking the HTML in the response from your HTTP request above. Hardcoded it to "proteins" because this will not change.)
    row = table.find_next('td') #use the beautifulsoup library again to find a html element in the found table. (Give me the first column value.)
    
    return row.text # row.text gives us the protein ID, we will need this protein ID to generate our table. (Example: VCAM1_HUMAN -> P19320)


def extract_table(proteinid):
    url = "http://cicblade.dep.usal.es:8080/APID/InteractionsGrid.action?protein1="+str(proteinid)+"&protein2=NA"

    payload = {}
    headers = {
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'Accept-Language': 'en-US,en;q=0.9',
    'Connection': 'keep-alive',
    'Cookie': 'JSESSIONID=086030D12C94A8248DA2B5B9A84C16FA; _ga=GA1.2.581619552.1696441740; _gid=GA1.2.818200239.1696441740; _ga_7JSDHY18SK=GS1.2.1696441740.1.1.1696442421.0.0.0',
    'Referer': 'http://cicblade.dep.usal.es:8080/APID/searchProtein.action',
    'Upgrade-Insecure-Requests': '1',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'
    }

    response = requests.request("GET", url, headers=headers, data=payload)
    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.find('table', id="interactions") #Find the table

    rows = table.find_all("tr") # Find all table rows
    interactions = [] #initialize empty list
    for idx, row in enumerate(rows): #loop over rows, keep index
        if idx == 0: # first row is the header, skip.
            pass
        else:
            try:
                data1 = row.find_all("td") # get column
                interaction = { # build intraction object
                "ProteinA": data1[0].get_text().strip(),
                "ProteinB": data1[1].get_text().strip(),
                "MethodType": data1[2].get_text().strip(),
                "Method": data1[3].get_text().strip(),
                "Publication": re.sub(' +', ' ',data1[4].get_text().strip().replace("\n", "")),
                "Source": data1[5].get_text().strip()
                }
                interactions.append(interaction) #append interaction object to result list
            except:
                pass
    return interactions #return the result list


proteinid = extract_protein_id("VCAM1_HUMAN")
results = extract_table(proteinid)
print(json.dumps(results, indent=4))

In [ ]:
# Make an intersecions diagram using the pyUpSet to check the interactors of VCAM1 
from upsetplot import UpSet


